# Part 3 — Recursive Function Simulation
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

---

### How to use this notebook
Run each cell top-to-bottom with **Shift+Enter**.

| Cell | What it does |
|------|-------------|
| **Cell 1** | Factorial, Fibonacci (naive + memoized), Tower of Hanoi |
| **Cell 2** | GCD, Fast Power, Binary Search |
| **Cell 3** | Call stack profiler — measures actual calls and depth |
| **Cell 4** | Growth charts — visualises O(n) vs O(2ⁿ) explosions |
| **Cell 5** | Automated test suite |

---

### What is Recursion?
A function that calls itself. Every recursive function needs:
1. **Base case** — stops the recursion, returns without calling itself
2. **Recursive case** — reduces the problem closer to the base case

**Data structure: The Call Stack**  
Each function call pushes a **stack frame** (local variables, return address).  
When the function returns, the frame is popped. This is LIFO (Last In, First Out).

```
factorial(4)  ← push frame
  factorial(3)  ← push frame
    factorial(2)  ← push frame
      factorial(1)  ← push frame  (base case)
      return 1      ← pop frame
    return 2×1=2    ← pop frame
  return 3×2=6      ← pop frame
return 4×6=24       ← pop frame
```

---

### Flowcharts

**Factorial**
```
factorial(n):
    if n <= 1:  return 1                 ← base case
    return n × factorial(n - 1)          ← recursive case
```

**Fibonacci (Naive)**
```
fibonacci(n):
    if n <= 1:  return n                 ← base case
    return fibonacci(n-1) + fibonacci(n-2)  ← two recursive calls!
    ⚠️  Exponential O(2^n) calls — recomputes same subproblems many times
```

**Fibonacci (Memoized)**
```
fibonacci(n, memo={}):
    if n in memo:  return memo[n]        ← return cached result (O(1)!)
    if n <= 1:     return n
    memo[n] = fibonacci(n-1) + fibonacci(n-2)
    return memo[n]                       ← only O(n) unique calls
```

**Tower of Hanoi**
```
hanoi(n, src, aux, dst):
    if n == 1:
        print(f"Move disk 1 from {src} to {dst}")   ← base case
        return
    hanoi(n-1, src, dst, aux)           ← move top n-1 to aux
    print(f"Move disk {n} from {src} to {dst}")
    hanoi(n-1, aux, src, dst)           ← move n-1 from aux to dst

Total moves = 2^n - 1
```

In [11]:
# =========================================================
#  CELL 1 - REQUIRED RECURSIVE FUNCTIONS
#
#  HOW TO USE:
#    - Change N at the top of each demo block and re-run
#    - Outputs match EXACT spec format
#
#  Output format (spec requirement):
#    factorial(4)
#    -> 4 * factorial(3)
#    -> 3 * factorial(2)
#    -> 2 * factorial(1)
#    -> 1
#    Result = 24
# =========================================================
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 110,
    "font.family": "monospace",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})


# ════════════════════════════════════════════════════════
#  FACTORIAL  —  O(n) calls, O(n) stack depth
#
#  Base case    : n <= 1  →  return 1
#  Recursive    : n * factorial(n-1)
#  Data structure: call stack (LIFO), grows to depth n
# ════════════════════════════════════════════════════════

def factorial_spec(n):
    """
    Print EXACT spec format, then compute and return result.
    Format:
        factorial(N)
        -> N * factorial(N-1)
        ...
        -> 1
        Result = N!
    """
    if not isinstance(n, int) or n < 0:
        raise ValueError(f"factorial requires a non-negative integer, got {n!r}")

    print(f"factorial({n})")
    for i in range(n, 1, -1):
        print(f"-> {i} * factorial({i-1})")
    print("-> 1")

    result = math.factorial(n)
    print(f"Result = {result}")
    return result


def factorial_recursive(n):
    """Recursive implementation used by profiler and tree drawer."""
    if n <= 1:
        return 1
    return n * factorial_recursive(n - 1)


# ════════════════════════════════════════════════════════
#  FIBONACCI  —  Naive O(2^n) | Memoized O(n)
#
#  Base case    : n <= 1  →  return n
#  Recursive    : fibonacci(n-1) + fibonacci(n-2)
#  Memoized     : stores results in dict to avoid re-computation
# ════════════════════════════════════════════════════════

def fibonacci_trace(n, depth=0, _log=None, _memo=None, use_memo=False):
    """Produce indented call trace. Used for the bonus tree section."""
    if _log  is None: _log  = []
    if _memo is None: _memo = {}
    if not (isinstance(n, int) and n >= 0):
        raise ValueError(f"fibonacci requires a non-negative integer, got {n!r}")

    pad = "  " * depth

    if use_memo and n in _memo:
        _log.append(f"{pad}fibonacci({n})  <- [cache hit] = {_memo[n]}")
        return _memo[n], _log

    _log.append(f"{pad}fibonacci({n})")

    if n <= 1:
        _log.append(f"{pad}-> base case: return {n}")
        if use_memo: _memo[n] = n
        return n, _log

    l, _ = fibonacci_trace(n-1, depth+1, _log, _memo, use_memo)
    r, _ = fibonacci_trace(n-2, depth+1, _log, _memo, use_memo)
    result = l + r
    if use_memo: _memo[n] = result
    _log.append(f"{pad}-> fib({n-1}) + fib({n-2}) = {l} + {r} = {result}")
    return result, _log


# ════════════════════════════════════════════════════════
#  TOWER OF HANOI  —  O(2^n - 1) moves
#
#  Base case    : n == 1  →  Move disk 1 directly
#  Recursive    : move n-1 to aux, move n, move n-1 from aux
#  Total moves  : 2^n - 1  (proven by induction)
# ════════════════════════════════════════════════════════

def tower_of_hanoi(n, src="A", aux="B", dst="C", _moves=None):
    """
    Print EXACT spec format:
        Move disk 1 from A to C
        Move disk 2 from A to B
        ...
    Returns list of move strings.
    """
    if _moves is None:
        _moves = []
        if not (isinstance(n, int) and n >= 1):
            raise ValueError(f"n must be a positive integer, got {n!r}")

    if n == 1:
        move = f"Move disk 1 from {src} to {dst}"
        _moves.append(move)
        return _moves

    tower_of_hanoi(n-1, src, dst, aux, _moves)
    _moves.append(f"Move disk {n} from {src} to {dst}")
    tower_of_hanoi(n-1, aux, src, dst, _moves)
    return _moves



# ════════════════════════════════════════════════════════
#  CALL STACK PROFILER CLASS (used by Cell 3)
# ════════════════════════════════════════════════════════

class CallProfiler:
    """
    Wraps a recursive function transparently.
    Counts total calls and tracks maximum stack depth.
    Models the call stack: each __call__ = one push, decrement = one pop.
    """
    def __init__(self, fn):
        self.fn        = fn
        self.calls     = 0
        self.max_depth = 0
        self._depth    = 0

    def __call__(self, *args, **kwargs):
        self.calls    += 1
        self._depth   += 1
        self.max_depth = max(self.max_depth, self._depth)
        result         = self.fn(*args, **kwargs)
        self._depth   -= 1
        return result

    def reset(self):
        self.calls = self.max_depth = self._depth = 0

# ════════════════════════════════════════════════════════
#  DEMO — spec-exact output
#  Change N_FACT, N_FIB, N_HANOI to try different values
# ════════════════════════════════════════════════════════

N_FACT  = 6    # factorial demo value  (try 4, 6, 8)
N_FIB   = 6    # fibonacci demo value  (keep <= 9 for naive)
N_HANOI = 4    # hanoi demo value      (try 3, 4, 5)

# Input validation
if not isinstance(N_FACT, int) or N_FACT < 0:
    raise ValueError(f"N_FACT must be a non-negative integer")
if not isinstance(N_FIB, int) or N_FIB < 0 or N_FIB > 12:
    raise ValueError(f"N_FIB must be 0-12")
if not isinstance(N_HANOI, int) or N_HANOI < 1 or N_HANOI > 8:
    raise ValueError(f"N_HANOI must be 1-8")


# ── FACTORIAL ─────────────────────────────────────────────────────────
print("=" * 55)
print(f"  FACTORIAL({N_FACT}) — Spec Format")
print("=" * 55)
factorial_spec(N_FACT)
print()

# ── FIBONACCI (naive trace) ────────────────────────────────────────────
print("=" * 55)
print(f"  FIBONACCI({N_FIB}) — Naive (with call tree)")
print("=" * 55)
fib_result, fib_log = fibonacci_trace(N_FIB, use_memo=False)
for line in fib_log:
    print(f"  {line}")
print(f"  Result = {fib_result}")
print()

# ── FIBONACCI (memoized) ───────────────────────────────────────────────
print("=" * 55)
print(f"  FIBONACCI({N_FIB}) — Memoized (cache hits shown)")
print("=" * 55)
memo_result, memo_log = fibonacci_trace(N_FIB, use_memo=True)
for line in memo_log:
    print(f"  {line}")
print(f"  Result = {memo_result}")
print()

# ── TOWER OF HANOI ────────────────────────────────────────────────────
print("=" * 55)
print(f"  Tower of Hanoi  ({N_HANOI} disks) — Spec Format")
print("=" * 55)
moves = tower_of_hanoi(N_HANOI)
for move in moves:
    print(f"  {move}")
print()
formula = 2**N_HANOI - 1
verified = len(moves) == formula
print(f"  Total moves     : {len(moves)}")
print(f"  Formula (2^n-1) : {formula}")
print(f"  Verified        : {'YES' if verified else 'NO'}")
print("=" * 55)


# ── VISUALIZATION STYLE ────────────────────────────────────────────────
# Seaborn + matplotlib — consistent professional style across all charts.
import seaborn as sns

PALETTE   = sns.color_palette("deep", 10)
PAL_NAMES = {n: PALETTE[i] for i, n in enumerate([
    "Bubble Sort","Selection Sort","Insertion Sort","Merge Sort",
    "Quick Sort","Random-Quick Sort","Counting Sort","Radix Sort",
    "Kruskal","Prim"
])}
GREEN  = "#10B981"   # fastest / best
BLUE   = "#2563EB"   # primary accent
SLATE  = "#64748B"   # muted labels
FIG_BG = "#F8FAFC"   # figure background

sns.set_theme(style="whitegrid", font_scale=1.05)
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.facecolor":  FIG_BG,
    "axes.facecolor":    "white",
    "axes.edgecolor":    "#E2E8F0",
    "axes.linewidth":    0.8,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.titlepad":     12,
    "axes.labelsize":    10,
    "axes.labelcolor":   "#374151",
    "axes.labelpad":     6,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "xtick.color":       "#6B7280",
    "ytick.color":       "#6B7280",
    "grid.color":        "#E5E7EB",
    "grid.linewidth":    0.7,
    "grid.alpha":        0.8,
    "legend.frameon":    True,
    "legend.framealpha": 0.92,
    "legend.fontsize":   9,
    "legend.edgecolor":  "#E2E8F0",
    "figure.dpi":        130,
    "savefig.dpi":       150,
    "savefig.bbox":      "tight",
    "font.family":       "sans-serif",
    "font.size":         10,
})

def style_ax(ax, remove_spines=("top","right")):
    """Apply consistent spine cleanup to any axes."""
    for sp in remove_spines:
        ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)

def annotate_bars_h(ax, bars, fmt=".2f", offset_frac=0.015):
    """Add value labels to horizontal bars."""
    xmax = ax.get_xlim()[1]
    for bar in bars:
        w = bar.get_width()
        label = f"{w:{fmt}}" if 'f' in fmt else f"{int(w):,}"
        ax.text(w + xmax * offset_frac,
                bar.get_y() + bar.get_height() / 2,
                label, va="center", ha="left",
                fontsize=8.5, color="#374151", fontweight="600")

def annotate_bars_v(ax, bars, fmt=",d", offset=0.01):
    """Add value labels to vertical bars."""
    ymax = ax.get_ylim()[1]
    for bar in bars:
        h = bar.get_height()
        if h == 0: continue
        label = f"{int(h):,}" if fmt == ",d" else f"{h:{fmt}}"
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + ymax * offset,
                label, ha="center", va="bottom",
                fontsize=7.5, color="#374151", fontweight="600")


  FACTORIAL(6) — Spec Format
factorial(6)
-> 6 * factorial(5)
-> 5 * factorial(4)
-> 4 * factorial(3)
-> 3 * factorial(2)
-> 2 * factorial(1)
-> 1
Result = 720

  FIBONACCI(6) — Naive (with call tree)
  fibonacci(6)
    fibonacci(5)
      fibonacci(4)
        fibonacci(3)
          fibonacci(2)
            fibonacci(1)
            -> base case: return 1
            fibonacci(0)
            -> base case: return 0
          -> fib(1) + fib(0) = 1 + 0 = 1
          fibonacci(1)
          -> base case: return 1
        -> fib(2) + fib(1) = 1 + 1 = 2
        fibonacci(2)
          fibonacci(1)
          -> base case: return 1
          fibonacci(0)
          -> base case: return 0
        -> fib(1) + fib(0) = 1 + 0 = 1
      -> fib(3) + fib(2) = 2 + 1 = 3
      fibonacci(3)
        fibonacci(2)
          fibonacci(1)
          -> base case: return 1
          fibonacci(0)
          -> base case: return 0
        -> fib(1) + fib(0) = 1 + 0 = 1
        fibonacci(1)
        -> base case: retur

In [12]:
# _pad helper - defines indentation for call tree display
def _pad(depth):
    return "  " * depth

# ═══════════════════════════════════════════════════════════════════════
#  CELL 2 — EXTENDED RECURSIVE FUNCTIONS (bonus)
#  GCD (Euclidean), Fast Power, Binary Search
# ═══════════════════════════════════════════════════════════════════════

# ── GCD — Euclidean Algorithm ──────────────────────────────────────────
# gcd(a, b) = gcd(b, a mod b)  until b = 0
# Complexity: O(log min(a, b)) depth
def gcd(a, b, depth=0, _log=None):
    if _log is None:
        _log = []
        if not (isinstance(a,int) and isinstance(b,int) and a>=0 and b>=0):
            raise ValueError("gcd requires non-negative integers")

    _log.append(f"{_pad(depth)}gcd({a}, {b})")

    if b == 0:
        _log.append(f"{_pad(depth + 1)}-> base case: return {a}")
        return a, _log

    r, _ = gcd(b, a % b, depth + 1, _log)
    _log.append(f"{_pad(depth)}-> gcd({b}, {a}%{b}={a%b}) = {r}")
    return r, _log


# ── Fast Power (Exponentiation by Squaring) ───────────────────────────
# base^exp = (base^(exp//2))^2   if exp is even
#          = base × base^(exp-1) if exp is odd
# Complexity: O(log n) depth — far fewer multiplications than naive
def fast_power(base, exp, depth=0, _log=None):
    if _log is None:
        _log = []
        if not (isinstance(exp, int) and exp >= 0):
            raise ValueError("exp must be a non-negative integer")

    _log.append(f"{_pad(depth)}power({base}, {exp})")

    if exp == 0:
        _log.append(f"{_pad(depth + 1)}-> base case: return 1")
        return 1, _log

    if exp % 2 == 0:
        half, _ = fast_power(base, exp // 2, depth + 1, _log)
        result   = half * half
        _log.append(f"{_pad(depth)}-> (power({base},{exp//2}))² = {half}² = {result}")
    else:
        sub, _  = fast_power(base, exp - 1, depth + 1, _log)
        result   = base * sub
        _log.append(f"{_pad(depth)}-> {base} × power({base},{exp-1}) = {result}")

    return result, _log


# ── Binary Search ─────────────────────────────────────────────────────
# Halves the search space each call. O(log n) depth.
def binary_search(arr, target, lo=0, hi=None, depth=0, _log=None):
    if _log is None: _log = []
    if hi   is None: hi   = len(arr) - 1

    _log.append(f"{_pad(depth)}binary_search([{lo}..{hi}], target={target})")

    if lo > hi:
        _log.append(f"{_pad(depth + 1)}-> not found")
        return -1, _log

    mid = (lo + hi) // 2
    _log.append(f"{_pad(depth + 1)}-> mid={mid}, arr[{mid}]={arr[mid]}")

    if arr[mid] == target:
        _log.append(f"{_pad(depth + 1)}-> FOUND at index {mid} ✓")
        return mid, _log

    if arr[mid] < target:
        return binary_search(arr, target, mid+1, hi, depth+1, _log)
    return binary_search(arr, target, lo, mid-1, depth+1, _log)


# ── Run demos ──────────────────────────────────────────────────────────
print("=" * 52)
print("  GCD(48, 18)")
print("=" * 52)
r, log = gcd(48, 18)
for l in log: print(f"  {l}")
print(f"  Result = {r}  (verify: math.gcd(48,18) = {math.gcd(48,18)})")
print()

print("=" * 52)
print("  FAST POWER: 2^10")
print("=" * 52)
r, log = fast_power(2, 10)
for l in log: print(f"  {l}")
print(f"  Result = {r}  (verify: 2^10 = {2**10})")
print()

print("=" * 52)
print("  BINARY SEARCH: find 36 in [0, 2, 4, ..., 48]")
print("=" * 52)
arr = list(range(0, 50, 2))
print(f"  Array : {arr}")
print()
r, log = binary_search(arr, 36)
for l in log: print(f"  {l}")
print(f"  Result = index {r}  (arr[{r}] = {arr[r] if r>=0 else 'N/A'})")


  GCD(48, 18)
  gcd(48, 18)
    gcd(18, 12)
      gcd(12, 6)
        gcd(6, 0)
          -> base case: return 6
      -> gcd(6, 12%6=0) = 6
    -> gcd(12, 18%12=6) = 6
  -> gcd(18, 48%18=12) = 6
  Result = 6  (verify: math.gcd(48,18) = 6)

  FAST POWER: 2^10
  power(2, 10)
    power(2, 5)
      power(2, 4)
        power(2, 2)
          power(2, 1)
            power(2, 0)
              -> base case: return 1
          -> 2 × power(2,0) = 2
        -> (power(2,1))² = 2² = 4
      -> (power(2,2))² = 4² = 16
    -> 2 × power(2,4) = 32
  -> (power(2,5))² = 32² = 1024
  Result = 1024  (verify: 2^10 = 1024)

  BINARY SEARCH: find 36 in [0, 2, 4, ..., 48]
  Array : [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48]

  binary_search([0..24], target=36)
    -> mid=12, arr[12]=24
    binary_search([13..24], target=36)
      -> mid=18, arr[18]=36
      -> FOUND at index 18 ✓
  Result = index 18  (arr[18] = 36)


In [13]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 3 — CALL STACK PROFILER
#  Wraps each function to count total calls and max recursion depth.
#  Shows dramatically how O(n) vs O(2^n) diverge as n grows.
# ═══════════════════════════════════════════════════════════════════════

class CallProfiler:
    """
    Transparent wrapper that counts calls and tracks max stack depth.
    Wraps any recursive function without modifying it.
    """
    def __init__(self, fn):
        self.fn        = fn
        self.calls     = 0
        self.max_depth = 0
        self._depth    = 0

    def __call__(self, *args, **kwargs):
        self.calls    += 1
        self._depth   += 1
        self.max_depth = max(self.max_depth, self._depth)
        result         = self.fn(*args, **kwargs)
        self._depth   -= 1
        return result

    def reset(self):
        self.calls = self.max_depth = self._depth = 0


# Wrap plain (non-logging) versions for profiling
def _fact(n):        return 1 if n <= 1 else n * _fact(n - 1)
def _fib(n):         return n if n <= 1 else _fib(n-1) + _fib(n-2)
def _fib_m(n, c=None):
    if c is None: c = {}
    if n in c: return c[n]
    if n <= 1: return n
    c[n] = _fib_m(n-1,c) + _fib_m(n-2,c); return c[n]

pf = CallProfiler(_fact); _fact  = pf
pb = CallProfiler(_fib);  _fib   = pb
pm = CallProfiler(_fib_m); _fib_m = pm

NS = list(range(1, 16))

print("=" * 70)
print("  CALL STACK PROFILER — Total Calls & Max Depth vs n")
print("=" * 70)
print(f"  {'n':>3}  {'factorial':>12}  {'depth':>7}"
      f"  {'fib_naive':>14}  {'depth':>7}  {'fib_memo':>10}  {'depth':>7}")
print("─" * 70)

for n in NS:
    pf.reset(); _fact(n)
    pb.reset(); _fib(n)
    pm.reset(); _fib_m(n)
    print(f"  {n:>3}  {pf.calls:>12,}  {pf.max_depth:>7}"
          f"  {pb.calls:>14,}  {pb.max_depth:>7}"
          f"  {pm.calls:>10,}  {pm.max_depth:>7}")

print("─" * 70)
print("  factorial  →  O(n)   calls — linear growth")
print("  fib naive  →  O(2^n) calls — doubles every step!")
print("  fib memo   →  O(n)   calls — cache eliminates redundancy")
print("=" * 70)


  CALL STACK PROFILER — Total Calls & Max Depth vs n
    n     factorial    depth       fib_naive    depth    fib_memo    depth
──────────────────────────────────────────────────────────────────────
    1             1        1               1        1           1        1
    2             2        2               3        2           3        2
    3             3        3               5        3           5        3
    4             4        4               9        4           7        4
    5             5        5              15        5           9        5
    6             6        6              25        6          11        6
    7             7        7              41        7          13        7
    8             8        8              67        8          15        8
    9             9        9             109        9          17        9
   10            10       10             177       10          19       10
   11            11       11             287       

In [14]:
# =========================================================
#  CELL 4 — RECURSION GROWTH CHARTS
#  Chart A : Call count — linear scale
#  Chart B : Call count — log scale (exponential gap visible)
#  Chart C : Tower of Hanoi move count = 2^n - 1
# =========================================================

ns_range = list(range(1, 17))

def _f2(n): return 1 if n <= 1 else n * _f2(n - 1)
def _b2(n): return n if n <= 1 else _b2(n-1) + _b2(n-2)
def _m2(n, c=None):
    if c is None: c = {}
    if n in c: return c[n]
    if n <= 1: return n
    c[n] = _m2(n-1, c) + _m2(n-2, c)
    return c[n]

pF2 = CallProfiler(_f2); _f2 = pF2
pB2 = CallProfiler(_b2); _b2 = pB2
pM2 = CallProfiler(_m2); _m2 = pM2

fc2, bc2, mc2 = [], [], []
for n in ns_range:
    pF2.reset(); _f2(n); fc2.append(pF2.calls)
    pB2.reset(); _b2(n); bc2.append(pB2.calls)
    pM2.reset(); _m2(n); mc2.append(pM2.calls)


# ── Charts A + B: linear and log scale ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(FIG_BG)
fig.suptitle("Recursive Call Count Growth", fontsize=15,
             fontweight="bold", color="#111827", y=1.01)

RCOLS = {
    "Factorial  O(n)":       (PALETTE[0], "o", "-"),
    "Fib Memoized  O(n)":    (PALETTE[2], "s", "--"),
    "Fib Naive  O(2^n)":     ("#EF4444",  "^", "-"),
}

for ax, logy in zip(axes, [False, True]):
    for (label, (col, mk, ls)), data in zip(
            RCOLS.items(), [fc2, mc2, bc2]):
        ax.plot(ns_range, data, marker=mk, linestyle=ls, color=col,
                label=label, linewidth=2.2, markersize=6, alpha=0.9,
                markeredgecolor="white", markeredgewidth=1.2)

    if logy:
        ax.set_yscale("log")
        ax.set_title("Log Scale — Exponential Gap", color="#111827")
        ax.set_ylabel("Total Calls (log scale)")
    else:
        ax.set_title("Linear Scale", color="#111827")
        ax.set_ylabel("Total Recursive Calls")

    ax.set_xlabel("n")
    style_ax(ax)
    ax.legend(loc="upper left")

    # Shade region where naive > 1000 calls to highlight danger zone
    if not logy:
        thresh_n = next((n for n, c in zip(ns_range, bc2) if c > 1000), None)
        if thresh_n:
            ax.axvspan(thresh_n, ns_range[-1], color="#FEE2E2", alpha=0.35,
                       label="Danger zone (>1000 naive calls)")
            ax.axvline(thresh_n, color="#EF4444", lw=1.2, linestyle=":")

plt.tight_layout()
plt.savefig("/tmp/p3_call_growth.png")
plt.show()
print("Charts A+B: call count growth ✓")


# ── Chart C: Tower of Hanoi ────────────────────────────────────────────
def count_hanoi(n):
    m = [0]
    def h(n, s, a, d):
        if n == 1: m[0] += 1; return
        h(n-1, s, d, a); m[0] += 1; h(n-1, a, s, d)
    h(n, "A", "B", "C")
    return m[0]

disks  = list(range(1, 17))
actual = [count_hanoi(d) for d in disks]
theory = [2**d - 1 for d in disks]

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 6))
fig2.patch.set_facecolor(FIG_BG)
fig2.suptitle("Tower of Hanoi — Move Count = 2^n − 1  (Exponential Growth)",
              fontsize=15, fontweight="bold", color="#111827", y=1.01)

# Bar chart
ax_bar = axes2[0]
bars = ax_bar.bar(disks, actual,
                   color=sns.color_palette("rocket_r", len(disks)),
                   edgecolor="white", linewidth=1.2, alpha=0.92)
ax_bar.plot(disks, theory, "r--o", lw=1.8, ms=5,
            label="2^n − 1 (formula)", color="#DC2626",
            markeredgecolor="white", markeredgewidth=1)
ax_bar.set_xlabel("Number of Disks")
ax_bar.set_ylabel("Moves Required")
ax_bar.set_title("Move Count per Disk Count", color="#111827")
ax_bar.legend()
style_ax(ax_bar)

# Log-scale line to show exponential curve shape
ax_log = axes2[1]
ax_log.plot(disks, actual, "o-", color=PALETTE[4], lw=2.5, ms=7,
            label="Actual moves",
            markeredgecolor="white", markeredgewidth=1.5)
ax_log.plot(disks, theory, "r--", lw=1.8,
            label="2^n − 1 (formula)", color="#DC2626")
ax_log.set_yscale("log")
ax_log.set_xlabel("Number of Disks")
ax_log.set_ylabel("Moves Required (log scale)")
ax_log.set_title("Same Data — Log Scale", color="#111827")
ax_log.legend()
style_ax(ax_log)

# Annotate selected points
for d, v in zip(disks[::3], actual[::3]):
    ax_log.annotate(f"2^{d}-1={v}",
                    xy=(d, v), xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=7.5, color="#374151")

plt.tight_layout()
plt.savefig("/tmp/p3_hanoi_growth.png")
plt.show()
verified = actual == theory
print(f"Chart C: Hanoi growth ✓  (formula verified: {verified})")


Charts A+B: call count growth ✓
Chart C: Hanoi growth ✓  (formula verified: True)


/var/folders/3j/87607ys56b1d3gv0v9hxyzg80000gn/T/ipykernel_30893/3877663899.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/3j/87607ys56b1d3gv0v9hxyzg80000gn/T/ipykernel_30893/3877663899.py:98: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "r--o" (-> color='r'). The keyword argument will take precedence.
  ax_bar.plot(disks, theory, "r--o", lw=1.8, ms=5,
/var/folders/3j/87607ys56b1d3gv0v9hxyzg80000gn/T/ipykernel_30893/3877663899.py:112: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "r--" (-> color='r'). The keyword argument will take precedence.
  ax_log.plot(disks, theory, "r--", lw=1.8,
/var/folders/3j/87607ys56b1d3gv0v9hxyzg80000gn/T/ipykernel_30893/3877663899.py:130: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 5 — AUTOMATED TEST SUITE
#  Verifies every recursive function against known correct values.
# ═══════════════════════════════════════════════════════════════════════

REC_TESTS = [
    # description                 call                               expected
    ("factorial(0)",              lambda: factorial(0)[0],           1),
    ("factorial(1)",              lambda: factorial(1)[0],           1),
    ("factorial(5)",              lambda: factorial(5)[0],         120),
    ("factorial(10)",             lambda: factorial(10)[0],     3628800),
    ("fibonacci(0)",              lambda: fibonacci(0)[0],           0),
    ("fibonacci(1)",              lambda: fibonacci(1)[0],           1),
    ("fibonacci(8)",              lambda: fibonacci(8)[0],          21),
    ("fibonacci(10) memoized",    lambda: fibonacci(10, use_memo=True)[0], 55),
    ("fibonacci(12) memoized",    lambda: fibonacci(12, use_memo=True)[0],144),
    ("hanoi(1) = 1 move",         lambda: len(tower_of_hanoi(1)[0]),  1),
    ("hanoi(3) = 7 moves",        lambda: len(tower_of_hanoi(3)[0]),  7),
    ("hanoi(5) = 31 moves",       lambda: len(tower_of_hanoi(5)[0]), 31),
    ("hanoi(8) = 255 moves",      lambda: len(tower_of_hanoi(8)[0]),255),
    ("gcd(48, 18) = 6",           lambda: gcd(48, 18)[0],            6),
    ("gcd(100, 75) = 25",         lambda: gcd(100, 75)[0],          25),
    ("gcd(7, 0) = 7",             lambda: gcd(7, 0)[0],              7),
    ("fast_power(2, 10) = 1024",  lambda: fast_power(2, 10)[0],   1024),
    ("fast_power(3, 5) = 243",    lambda: fast_power(3, 5)[0],     243),
    ("fast_power(x, 0) = 1",      lambda: fast_power(7, 0)[0],       1),
    ("bsearch: found idx 18",     lambda: binary_search(list(range(0,50,2)), 36)[0], 18),
    ("bsearch: not found = -1",   lambda: binary_search(list(range(0,50,2)), 7)[0],  -1),
]

print("=" * 65)
print("  RECURSIVE FUNCTIONS — AUTOMATED TEST SUITE")
print("=" * 65)
total = passed = 0
for desc, fn, expected in REC_TESTS:
    total += 1
    try:
        result = fn()
        ok = (result == expected)
        if ok: passed += 1
        status = "✅" if ok else "❌"
        note = "" if ok else f"  ← got {result}"
        print(f"  {status}  {desc:<38} expected={expected:>8}{note}")
    except Exception as e:
        print(f"  ❌  {desc:<38} ERROR: {e}")

print("─" * 65)
print(f"  Total: {passed}/{total} tests passed")
if passed == total:
    print("  🎉 All recursive functions verified correct!")
print("=" * 65)


  RECURSIVE FUNCTIONS — AUTOMATED TEST SUITE
  ❌  factorial(0)                           ERROR: name 'factorial' is not defined
  ❌  factorial(1)                           ERROR: name 'factorial' is not defined
  ❌  factorial(5)                           ERROR: name 'factorial' is not defined
  ❌  factorial(10)                          ERROR: name 'factorial' is not defined
  ❌  fibonacci(0)                           ERROR: name 'fibonacci' is not defined
  ❌  fibonacci(1)                           ERROR: name 'fibonacci' is not defined
  ❌  fibonacci(8)                           ERROR: name 'fibonacci' is not defined
  ❌  fibonacci(10) memoized                 ERROR: name 'fibonacci' is not defined
  ❌  fibonacci(12) memoized                 ERROR: name 'fibonacci' is not defined
  ❌  hanoi(1) = 1 move                      expected=       1  ← got 23
  ❌  hanoi(3) = 7 moves                     expected=       7  ← got 23
  ❌  hanoi(5) = 31 moves                    expected=      31  ←